## The rest of the workload family

Deployments cover stateless services — the common case. Four other controllers each fit a specific shape.

**DaemonSet — *one pod per node.*** Exactly one copy on every matching node. Add a node → a pod appears; remove it → gone. For **node-level agents**: log shippers (Fluent Bit), metrics agents (`node-exporter`), CNI/CSI plugins, security agents (Falco). You'll see them in `kube-system` — `kube-proxy` and the CNI plugin both run as DaemonSets.

**StatefulSet — *identity + storage that survive restarts.*** A Deployment treats pods as interchangeable; a StatefulSet gives ordinals — `pod-0`, `pod-1` — each with a **stable name**, **stable network identity**, and **its own PersistentVolume that follows it**. For **stateful systems**: databases, Cassandra/Kafka/Elasticsearch, etcd/Zookeeper. Identity comes via a headless Service (notebook 04), storage via `volumeClaimTemplates` (notebook 06).

**Job — *run a pod to completion.*** Creates pods and watches until they exit `0`, retrying per `backoffLimit`. For **batch work**: migrations, reports, ETL. Defaults to `restartPolicy: OnFailure`; `parallelism` and `completions` control how many run and how many must succeed.

**CronJob — *a Job on a schedule.*** `schedule: "0 2 * * *"` + a `jobTemplate`. For **periodic** work: backups, log rotation, cache warmers.

### Picking one

```
runs forever? ──no──> once → Job   |   scheduled → CronJob
   │yes
   ├─ needs stable identity + per-pod storage? → StatefulSet
   ├─ one copy per node? → DaemonSet
   └─ else → Deployment
```

For the CKA, **Deployment** is the answer most of the time — reach for the others only when the question describes their exact shape. On our map, all five live in the **Workload kinds** box; the ownership arrows (Deployment→ReplicaSet→Pod, CronJob→Job→Pod) are the wiring to memorise.